In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob


Automatic pdb calling has been turned OFF


/home/halechr/FastData/Bapun/RatS/NeuroPy/neuropy/utils/mixins/time_slicing.py:404: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("time_point_event")


In [3]:
import spikeinterface.full as si


## Bapun Format:
basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# basedir = Path('/Volumes/iNeo/Data/Bapun/Day5TwoNovel') # MacOS

# n_channels: int = 200
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day1Openfield'
excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']

xml_path: Path = find_first_file_rglob(basedir, '*.xml', recursive=False)
xml_path
print(f'xml_path: {xml_path}')


xml_path: /home/halechr/FastData/Bapun/RatS/Day1Openfield/RatS-Day1Openfield.xml


In [4]:
concatenated_dat_file: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.dat").resolve())
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."


concatenated_dat_file: /home/halechr/FastData/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield.dat


In [5]:
spiking_circus_dir: Path = windows_to_wsl_path_if_needed(basedir.joinpath('spyk-circ', basename).resolve())
assert spiking_circus_dir.exists(), f"spiking_circus_dir: {spiking_circus_dir} does not exist!"
# 1. Map the raw binary file (doesn't load into RAM yet)
# se.read_phy
spiking_circus_loaded = se.read_spykingcircus(spiking_circus_dir)

spiking_circus_loaded

SpykingCircusSortingExtractor: 138 units - 1 segments - 30.0kHz

In [ ]:
# spiking_circus_loaded

si.create_sorting_analyzer()

In [ ]:
n_channels

In [6]:
recording = se.read_binary(
    file_paths=concatenated_dat_file.as_posix(),
    sampling_frequency=dat_file_sampling_rate,
    num_channels=n_channels,
    dtype="int16",
)
recording

BinaryRecordingExtractor: 195 channels - 30.0kHz - 1 segments - 131,122,002 samples 
                          4,370.73s (1.21 hours) - int16 dtype - 47.63 GiB
  file_paths: ['/home/halechr/FastData/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield.dat']

In [7]:
from probeinterface.io import read_prb

prb_path: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.prb").resolve())
probe = read_prb(prb_path).probes[0]
recording = recording.set_probe(probe, in_place=False)
recording_filtered = si.bandpass_filter(recording)
recording_filtered


BandpassFilterRecording: 126 channels - 30.0kHz - 1 segments - 131,122,002 samples 
                         4,370.73s (1.21 hours) - int16 dtype - 30.77 GiB

In [8]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

ValueError: Folder /home/halechr/FastData/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield-sorting_analyzer already exists! Use overwrite=True to overwrite it.

In [9]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')


sorting_outputs_folder: /home/halechr/FastData/Bapun/RatS/Day1Openfield/SORTING


In [ ]:

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [11]:
# run Tridesclous
TDC_Output_Path: Path = sorting_outputs_folder.joinpath("folder_TDC")
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording_filtered, folder=TDC_Output_Path)
sorting_TDC

/home/halechr/FastData/Bapun/RatS/bapun_sess_init_scripts/.venv/lib/python3.11/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


TridesclousSortingExtractor: 4 units - 1 segments - 30.0kHz

In [12]:
TDC_Final_Sorting_Output_Path: Path = TDC_Output_Path.joinpath('tridescloud_sorting_output')
sorting_TDC.save(folder=TDC_Final_Sorting_Output_Path)


/home/halechr/FastData/Bapun/RatS/bapun_sess_init_scripts/.venv/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:384: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")


NumpyFolder (NumpyFolderSorting): 4 units - 1 segments - 30.0kHz

In [ ]:
# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder=sorting_outputs_folder.joinpath("folder_KS4"))

In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"))
sorting_SC2

In [ ]:
sorting_SC2


# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)